# MLP — Exercício 3

## Exercise 3

[Enunciado](https://insper.github.io/ann-dl/2026.2/exercises/mlp/)



In [1]:
from pathlib import Path
import sys
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

code_dir = Path.cwd()
if not (code_dir / "exercicio_3.py").is_file():
    code_dir = Path.cwd() / "docs" / "exercises" / "mlp" / "code"
if not (code_dir / "exercicio_3.py").is_file():
    raise FileNotFoundError("Abra o notebook na pasta code ou na raiz do repositório.")
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))
from mlp import MLP
from exercicio_2 import LAYERS as BINARY_LAYERS
from exercicio_3 import plot_loss, plot_confusion, most_confused_pairs

# Reconstitui o estado do gerador após a inicialização do Exercício 2.
rng = np.random.default_rng(42)
_previous_initialization = MLP(BINARY_LAYERS, rng)


### A — Generate the data

Foram usadas **1500 amostras, quatro características informativas e três classes**, com dois grupos por classe e `class_sep=1.0`. A geração usa exatamente os parâmetros do enunciado: sem características redundantes ou repetidas e com `random_state=42`.

A divisão estratificada reserva **1200 amostras para treino e 300 para teste**. As características são usadas na escala original. O teste participa apenas da avaliação final.

| Conjunto | Classe 0 | Classe 1 | Classe 2 | Total |
|---|---:|---:|---:|---:|
| Completo | 499 | 499 | 502 | 1500 |
| Treino | 399 | 399 | 402 | 1200 |
| Teste | 100 | 100 | 100 | 300 |

O gerador de pesos continua a sequência iniciada com semente 42 no Exercício 2. Na execução isolada deste exercício, a inicialização da rede binária é repetida apenas para reconstruir o estado do mesmo `rng`; o treinamento anterior não consome números aleatórios.


In [2]:
X, y = make_classification(
    n_samples=1500, n_features=4, n_informative=4,
    n_redundant=0, n_repeated=0, n_classes=3,
    n_clusters_per_class=2, class_sep=1.0, random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
print(f"Dados: {X.shape}; classes: {np.bincount(y)}")
print(f"Treino: {len(X_train)}; classes: {np.bincount(y_train)}")
print(f"Teste: {len(X_test)}; classes: {np.bincount(y_test)}")


Dados: (1500, 4); classes: [499 499 502]
Treino: 1200; classes: [399 399 402]
Teste: 300; classes: [100 100 100]


### B — Adapt the network

A classe `MLP` é a mesma do Exercício 2, importada de `mlp.py`. Os métodos **`forward`, `backward` e `update` foram reutilizados sem alterações**. A adaptação é feita passando três unidades de saída, a função `softmax` e a entropia cruzada categórica como argumentos.

#### Softmax e entropia cruzada categórica

Para uma amostra $i$, cada logit $u_{ik}$ gera uma probabilidade para a classe $k$:

$$
p_{ik}=\frac{e^{u_{ik}}}{\sum_{r=1}^{3}e^{u_{ir}}}.
$$

As três probabilidades somam 1. No código, subtrair o maior logit da linha antes das exponenciais preserva essa razão e evita overflow.

O alvo é representado em one-hot: $y_{ik}=1$ para a classe correta e $0$ para as outras. A perda média é:

$$
L=-\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{3}y_{ik}\log(p_{ik}).
$$

O cálculo utiliza log-probabilidades obtidas dos logits, evitando calcular `log(0)` quando uma probabilidade é muito pequena.

#### Derivação do gradiente da saída

A derivada de uma probabilidade em relação a um logit é:

$$
\frac{\partial p_{ik}}{\partial u_{ij}}
=p_{ik}(\delta_{kj}-p_{ij}),
$$

onde $\delta_{kj}=1$ se $k=j$ e $0$ caso contrário. Como
$\partial L/\partial p_{ik}=-y_{ik}/(Np_{ik})$, pela regra da cadeia:

$$
\frac{\partial L}{\partial u_{ij}}
=\sum_{k=1}^{3}\frac{\partial L}{\partial p_{ik}}
\frac{\partial p_{ik}}{\partial u_{ij}}
=\sum_{k=1}^{3}\left(-\frac{y_{ik}}{Np_{ik}}\right)
p_{ik}(\delta_{kj}-p_{ij}).
$$

Cancelando $p_{ik}$ e separando os termos:

$$
\frac{\partial L}{\partial u_{ij}}
=-\frac{1}{N}\left[
\sum_{k=1}^{3}y_{ik}\delta_{kj}
-p_{ij}\sum_{k=1}^{3}y_{ik}\right].
$$

O primeiro somatório é $y_{ij}$, e o segundo vale 1 porque o alvo é one-hot. Portanto:

$$
\frac{\partial L}{\partial u_{ij}}
=-\frac{1}{N}(y_{ij}-p_{ij})
=\frac{p_{ij}-y_{ij}}{N}.
$$

Em forma matricial, **$\partial L/\partial\mathbf{U}=(\mathbf{P}-\mathbf{Y})/N$**.
Esse é o valor passado ao mesmo `backward` do exercício anterior; a média por $N$ é aplicada uma única vez.


In [3]:
def softmax(logits):
    """Converte os três logits em probabilidades que somam 1 por linha."""
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


def categorical_cross_entropy(logits, probabilities, y):
    """Retorna a perda média e a derivada (P-Y)/N em relação aos logits."""
    labels = np.asarray(y, dtype=int)
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_probabilities = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    loss = -np.mean(log_probabilities[np.arange(len(labels)), labels])
    targets = np.zeros_like(probabilities)
    targets[np.arange(len(labels)), labels] = 1.0
    delta = (probabilities - targets) / len(labels)
    return float(loss), delta


### C — Train and evaluate

Foi utilizada a arquitetura **4 → 16 → 3**, com uma camada oculta `tanh`, saída `softmax`, inicialização de Xavier e biases inicialmente zero.

| Configuração | Valor |
|---|---|
| Arquitetura | 4 entradas, 16 neurônios ocultos, 3 saídas |
| Parâmetros treináveis | 131 |
| Ativação oculta | tanh |
| Ativação de saída | softmax |
| Perda | Entropia cruzada categórica média |
| Taxa de aprendizado | 0,1 |
| Épocas | 2000 |
| Lote | Todas as 1200 amostras de treino |
| Predição | Classe com a maior probabilidade |

A arquitetura e os hiperparâmetros foram definidos antes de avaliar o teste. A perda de treino caiu de **1,0511 para 0,3383**.

A **acurácia no teste foi de 84,67%**, com **254 acertos em 300 amostras**. Na Figura 5, as linhas representam as classes reais e as colunas representam as classes previstas:

$$
\mathbf{C}=\begin{bmatrix}
86&6&8\\
11&78&11\\
5&5&90
\end{bmatrix}.
$$

A diagonal soma $86+78+90=254$ acertos; os valores fora da diagonal somam **46 erros**.


In [4]:
layer_sizes = [4, 16, 3]
learning_rate = 0.1
epochs = 2000
model = MLP(layer_sizes, rng, output_activation=softmax,
            loss_and_delta=categorical_cross_entropy)
history = model.fit(X_train, y_train, learning_rate=learning_rate, epochs=epochs)
predictions = model.predict(X_test)
mlp_accuracy = np.mean(predictions == y_test)
matrix = confusion_matrix(y_test, predictions, labels=[0, 1, 2])
print(f"Parâmetros: {model.parameter_count}")
print(f"Perda inicial: {history[0]:.4f}; perda final: {history[-1]:.4f}")
print(f"Acurácia da MLP: {mlp_accuracy:.2%}")
print(f"Acertos: {np.sum(predictions == y_test)} / {len(y_test)}")
print("Matriz de confusão (linhas reais; colunas previstas):")
print(matrix)


Parâmetros: 131
Perda inicial: 1.0511; perda final: 0.3383
Acurácia da MLP: 84.67%
Acertos: 254 / 300
Matriz de confusão (linhas reais; colunas previstas):
[[86  6  8]
 [11 78 11]
 [ 5  5 90]]


In [5]:
plot_loss(history)
plot_confusion(matrix)


![Figura 4 — Perda no treinamento multiclasse](../figures/fig4.png)

![Figura 5 — Matriz de confusão no teste](../figures/fig5.png)


### D — Analysis

O par mais confundido é **classes 0 e 1**, com **17 erros**: seis amostras reais da classe 0 foram previstas como classe 1, e onze da classe 1 foram previstas como classe 0. Somando as duas direções para cada par:

| Par de classes | Confusões no teste |
|---|---:|
| 0 e 1 | 6 + 11 = 17 |
| 0 e 2 | 8 + 5 = 13 |
| 1 e 2 | 11 + 5 = 16 |

A diferença entre 17 e 16 é pequena, indicando dificuldade semelhante nos pares 0–1 e 1–2. A classe 1 teve 78 acertos em 100 amostras, a menor proporção de acertos entre as três classes.

O conjunto possui **quatro dimensões informativas e seis grupos**, dois por classe. Cada classe pode ocupar mais de uma região nesse espaço; com `class_sep=1.0`, os grupos não são necessariamente bem separados. Uma explicação plausível para as confusões é a sobreposição entre regiões dos grupos dessas classes, que exige fronteiras mais complexas. A matriz informa onde ocorreram os erros; determinar a geometria exata dessa sobreposição exigiria analisar as quatro características.

A regressão logística, ajustada somente no treino, obteve **66,33% no teste** (**199 acertos em 300**). A MLP obteve **84,67%**, uma melhoria de **18,33 pontos percentuais**. A camada oculta acrescentou transformações não lineares das características, permitindo combinar várias regiões para representar as três classes e seus dois grupos por classe. A melhoria foi medida em dados que não participaram do ajuste.


In [6]:
baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train, y_train)
baseline_accuracy = np.mean(baseline.predict(X_test) == y_test)
print(f"Acurácia do baseline: {baseline_accuracy:.2%}")
print(f"Diferença: {100 * (mlp_accuracy - baseline_accuracy):.2f} pontos percentuais")
print("Par(es) mais confundido(s):", most_confused_pairs(matrix))


Acurácia do baseline: 66.33%
Diferença: 18.33 pontos percentuais
Par(es) mais confundido(s): [{'classes': [0, 1], 'count': 17}]


## Results summary

| Quantity | Value |
|---|---|
| Arquitetura | 4 → 16 → 3 |
| Parâmetros treináveis | 131 |
| Taxa de aprendizado e épocas | 0,1; 2000 |
| Acurácia da MLP no teste | 84,67% (254/300) |
| Acurácia do baseline no teste | 66,33% (199/300) |
| Perda de treino inicial e final | 1,0511; 0,3383 |
| Par mais confundido | Classes 0 e 1, com 17 confusões |
